In [1]:
import pandas as pd
import numpy as np


np.random.seed(42)
n = 891

pclass = np.random.choice([1, 2, 3], size=n, p=[216/891, 184/891, 491/891])
sex = np.random.choice(['male', 'female'], size=n, p=[577/891, 314/891])

age_mean = np.select([pclass == 1, pclass == 2, pclass == 3], [38, 30, 25])
age = np.random.normal(loc=age_mean, scale=13).clip(0.42, 80)

sibsp = np.random.choice([0, 1, 2, 3, 4, 5], size=n, p=[0.68, 0.23, 0.03, 0.02, 0.02, 0.02])
parch = np.random.choice([0, 1, 2, 3, 4], size=n, p=[0.76, 0.13, 0.08, 0.02, 0.01])

fare_mu = np.select([pclass == 1, pclass == 2, pclass == 3], [3.9, 2.9, 2.1])
fare = np.random.lognormal(mean=fare_mu, sigma=0.7)

embarked = np.random.choice(['S', 'C', 'Q'], size=n, p=[644/889, 168/889, 77/889])

has_cabin = np.random.rand(n) > 0.77
def pick_deck(pc):
    if pc == 1:
        return np.random.choice(list('ABCD'))
    elif pc == 2:
        return np.random.choice(list('DE'))
    else:
        return np.random.choice(list('FG'))
deck_pool = [pick_deck(pc) for pc in pclass]
cabin = [f"{d}{np.random.randint(1,100)}" if hc else np.nan for d, hc in zip(deck_pool, has_cabin)]

titles = np.where(sex == 'female', np.random.choice(['Mrs.', 'Miss.'], n), np.random.choice(['Mr.', 'Master.'], n))
name = [f"Passenger_{i} {t}" for i, t in enumerate(titles)]
ticket = [f"TCK{np.random.randint(10000,99999)}" for _ in range(n)]

logit = (-0.55
         + 2.6 * (sex == 'female')
         - 0.9 * (pclass == 3)
         - 0.35 * (pclass == 2)
         + 0.02 * (age < 12)
         + 0.004 * fare
         - 0.015 * age)
prob = 1 / (1 + np.exp(-logit))
survived = (np.random.rand(n) < prob).astype(int)

titanic_df = pd.DataFrame({
    'PassengerId': np.arange(1, n + 1), 'Survived': survived, 'Pclass': pclass,
    'Name': name, 'Sex': sex, 'Age': age, 'SibSp': sibsp, 'Parch': parch,
    'Ticket': ticket, 'Fare': fare, 'Cabin': cabin, 'Embarked': embarked,
})

age_missing_idx = np.random.choice(n, 177, replace=False)
titanic_df.loc[age_missing_idx, 'Age'] = np.nan
embarked_missing_idx = np.random.choice(n, 2, replace=False)
titanic_df.loc[embarked_missing_idx, 'Embarked'] = np.nan

print("Dataset Shape (rows, cols):", titanic_df.shape)
print("\nFirst 3 rows of raw dataset:")
print(titanic_df[['PassengerId','Survived','Pclass','Sex','Age','SibSp','Parch','Fare','Cabin','Embarked']].head(3))
print("\nMissing values per column:\n", titanic_df.isnull().sum()[titanic_df.isnull().sum() > 0])

Dataset Shape (rows, cols): (891, 12)

First 3 rows of raw dataset:
   PassengerId  Survived  Pclass     Sex        Age  SibSp  Parch       Fare  \
0            1         1       2  female        NaN      4      0  18.613864   
1            2         1       3  female  17.125706      0      0  10.988473   
2            3         0       3    male  34.647240      0      0   8.394804   

  Cabin Embarked  
0   NaN        C  
1   NaN        S  
2   NaN        C  

Missing values per column:
 Age         177
Cabin       694
Embarked      2
dtype: int64


In [2]:
from sklearn.impute import SimpleImputer

print("Missing values before imputation:\n", titanic_df.isnull().sum())

# Age: impute using the median Age within each Pclass/Sex group (more accurate than a global median)
titanic_df['Age'] = titanic_df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Embarked: impute using mode (most frequent port) since it is categorical with very few missing values
embarked_imputer = SimpleImputer(strategy='most_frequent')
titanic_df['Embarked'] = embarked_imputer.fit_transform(titanic_df[['Embarked']]).ravel()

# Cabin: over 75% missing, so raw values are dropped; presence/absence is kept as a binary indicator
titanic_df['HasCabin'] = titanic_df['Cabin'].notnull().astype(int)
titanic_df.drop(columns=['Cabin'], inplace=True)

print("\nMissing values after imputation:\n", titanic_df.isnull().sum())

Missing values before imputation:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          694
Embarked         2
dtype: int64

Missing values after imputation:
 PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
HasCabin       0
dtype: int64


In [3]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

# Encode categorical variables into numeric form
titanic_df['Sex_encoded'] = LabelEncoder().fit_transform(titanic_df['Sex'])          # male=1, female=0
titanic_df = pd.get_dummies(titanic_df, columns=['Embarked'], prefix='Embarked')      # one-hot encode port of embarkation
embarked_cols = [c for c in titanic_df.columns if c.startswith('Embarked_')]
titanic_df[embarked_cols] = titanic_df[embarked_cols].astype(int)

# Engineer FamilySize from SibSp + Parch
titanic_df['FamilySize'] = titanic_df['SibSp'] + titanic_df['Parch'] + 1

features = ['Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'HasCabin'] + embarked_cols
X = titanic_df[features]

# Apply MinMaxScaler (Normalization)
min_max_scaler = MinMaxScaler()
X_normalized = min_max_scaler.fit_transform(X)
print("Normalized Range Boundaries (Min/Max):", X_normalized.min(), "to", X_normalized.max())

# Apply StandardScaler (Standardization)
standard_scaler = StandardScaler()
X_standardized = standard_scaler.fit_transform(X)
print("Standardized Mean (approx 0):", round(X_standardized.mean(), 4))
print("Standardized Std Dev (approx 1):", round(X_standardized.std(), 4))

Normalized Range Boundaries (Min/Max): 0.0 to 1.0
Standardized Mean (approx 0): 0.0
Standardized Std Dev (approx 1): 1.0


In [4]:
from sklearn.preprocessing import PowerTransformer

original_skew = titanic_df['Fare'].skew()
print("Original skewness coefficient of 'Fare':", round(original_skew, 4))

fare_log = np.log1p(titanic_df['Fare'])
log_skew = fare_log.skew()
print("Skewness coefficient after Log Transformation (log1p):", round(log_skew, 4))

power_transformer = PowerTransformer(method='yeo-johnson')
fare_power = power_transformer.fit_transform(titanic_df[['Fare']])
power_skew = pd.Series(fare_power.flatten()).skew()
print("Skewness coefficient after Power Transformation:", round(power_skew, 4))

titanic_df['Fare_log'] = fare_log

Original skewness coefficient of 'Fare': 2.8206
Skewness coefficient after Log Transformation (log1p): 0.3575
Skewness coefficient after Power Transformation: 0.0371


In [5]:
processed_titanic_df = pd.DataFrame(data=X_standardized, columns=features)
processed_titanic_df['Survived'] = titanic_df['Survived'].values

processed_titanic_df.to_csv("processed_titanic.csv", index=False)
print("Processed Titanic dataset exported successfully to 'processed_titanic.csv'. Shape:", processed_titanic_df.shape)

Processed Titanic dataset exported successfully to 'processed_titanic.csv'. Shape: (891, 12)


In [6]:
from sklearn.preprocessing import StandardScaler

# Re-scale the full engineered feature set for dimensionality reduction
scaler2 = StandardScaler()
X_titanic_scaled = scaler2.fit_transform(titanic_df[features])

print("Standardized features shape:", X_titanic_scaled.shape)

Standardized features shape: (891, 11)


In [7]:
from sklearn.decomposition import PCA

pca_90 = PCA(n_components=0.90, random_state=42)
X_pca_90 = pca_90.fit_transform(X_titanic_scaled)

print("Original dimensions count:", X_titanic_scaled.shape[1])
print("Reduced dimensions count (to preserve 90% variance):", X_pca_90.shape[1])
print("Explained variance ratio per component:", np.round(pca_90.explained_variance_ratio_, 4))

Original dimensions count: 11
Reduced dimensions count (to preserve 90% variance): 7
Explained variance ratio per component: [0.1833 0.1712 0.169  0.1041 0.0959 0.0911 0.0886]


In [8]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

rf = RandomForestClassifier(random_state=42)
rf.fit(X, titanic_df['Survived'])

importances = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_}).sort_values(by='Importance', ascending=False)
print("Random Forest Feature Importances:\n", importances)

plt.figure(figsize=(6, 4))
sns.barplot(data=importances, x='Importance', y='Feature', palette='crest')
plt.title("Random Forest Feature Importances (Titanic)")
plt.tight_layout()
plt.savefig("rf_importance.png", dpi=100)
plt.close()

Random Forest Feature Importances:
         Feature  Importance
5          Fare    0.293144
1   Sex_encoded    0.256361
2           Age    0.239642
6    FamilySize    0.041350
0        Pclass    0.037798
3         SibSp    0.033160
7      HasCabin    0.029010
4         Parch    0.027137
10   Embarked_S    0.015967
8    Embarked_C    0.015126
9    Embarked_Q    0.011305


In [9]:
from sklearn.feature_selection import SelectKBest, chi2

# Chi-Square requires non-negative feature values, so the Min-Max normalized set is used
selector = SelectKBest(score_func=chi2, k=5)
selector.fit(X_normalized, titanic_df['Survived'])

chi2_scores = pd.DataFrame({'Feature': features, 'Chi2 Score': selector.scores_}).sort_values(by='Chi2 Score', ascending=False)
print("Chi-Square Feature Scores:\n", chi2_scores)

top5_selectkbest = chi2_scores.head(5)['Feature'].tolist()
print("\nTop 5 features selected by SelectKBest (Chi-Square):", top5_selectkbest)

Chi-Square Feature Scores:
         Feature  Chi2 Score
1   Sex_encoded   92.224252
0        Pclass    4.526861
5          Fare    2.619276
7      HasCabin    0.455638
8    Embarked_C    0.132857
9    Embarked_Q    0.122661
2           Age    0.109808
3         SibSp    0.061484
4         Parch    0.050808
10   Embarked_S    0.004718
6    FamilySize    0.002498

Top 5 features selected by SelectKBest (Chi-Square): ['Sex_encoded', 'Pclass', 'Fare', 'HasCabin', 'Embarked_C']


In [10]:
from sklearn.feature_selection import SelectKBest, chi2

# Chi-Square requires non-negative feature values, so the Min-Max normalized set is used
selector = SelectKBest(score_func=chi2, k=5)
selector.fit(X_normalized, titanic_df['Survived'])

chi2_scores = pd.DataFrame({'Feature': features, 'Chi2 Score': selector.scores_}).sort_values(by='Chi2 Score', ascending=False)
print("Chi-Square Feature Scores:\n", chi2_scores)

top5_selectkbest = chi2_scores.head(5)['Feature'].tolist()
print("\nTop 5 features selected by SelectKBest (Chi-Square):", top5_selectkbest)

Chi-Square Feature Scores:
         Feature  Chi2 Score
1   Sex_encoded   92.224252
0        Pclass    4.526861
5          Fare    2.619276
7      HasCabin    0.455638
8    Embarked_C    0.132857
9    Embarked_Q    0.122661
2           Age    0.109808
3         SibSp    0.061484
4         Parch    0.050808
10   Embarked_S    0.004718
6    FamilySize    0.002498

Top 5 features selected by SelectKBest (Chi-Square): ['Sex_encoded', 'Pclass', 'Fare', 'HasCabin', 'Embarked_C']


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train_t, y_test_t = train_test_split(X, titanic_df['Survived'], test_size=0.3, random_state=42, stratify=titanic_df['Survived'])

rf_full = RandomForestClassifier(random_state=42)
rf_full.fit(X_train, y_train_t)
preds_full = rf_full.predict(X_test)

top_4_cols = importances.head(4)['Feature'].tolist()
X_train_sub = X_train[top_4_cols]
X_test_sub = X_test[top_4_cols]

rf_sub = RandomForestClassifier(random_state=42)
rf_sub.fit(X_train_sub, y_train_t)
preds_sub = rf_sub.predict(X_test_sub)

print("--- CLASSIFICATION REPORT: ALL FEATURES ({} features) ---".format(len(features)))
print(classification_report(y_test_t, preds_full))

print("\n--- CLASSIFICATION REPORT: FEATURE-SELECTED SUBSET (Top 4 features:", top_4_cols, ") ---")
print(classification_report(y_test_t, preds_sub))

--- CLASSIFICATION REPORT: ALL FEATURES (11 features) ---
              precision    recall  f1-score   support

           0       0.77      0.88      0.82       165
           1       0.76      0.58      0.66       103

    accuracy                           0.77       268
   macro avg       0.77      0.73      0.74       268
weighted avg       0.77      0.77      0.76       268


--- CLASSIFICATION REPORT: FEATURE-SELECTED SUBSET (Top 4 features: ['Fare', 'Sex_encoded', 'Age', 'FamilySize'] ) ---
              precision    recall  f1-score   support

           0       0.76      0.87      0.81       165
           1       0.72      0.56      0.63       103

    accuracy                           0.75       268
   macro avg       0.74      0.71      0.72       268
weighted avg       0.75      0.75      0.74       268

